# RSNA: DINOv3 Small + 1750 MedGemma 27B soft labels, five folds

This is the matched supervised DINOv3 counterpart to the CoAtNet notebook: same 58 gold studies, same 1750-row **rsna_medgemma_modal27b_soft1000.csv**, folds, series selection, eight sampled centers per series, neighboring-slice channels, pooling, loss and best-validation-BCE selection. It uses pretrained DINOv3 features and fine-tunes the **last transformer block plus final LayerNorm**, with a trainable 12-label head. Earlier blocks are frozen. This notebook does not perform a new teacher/student self-supervised pretraining stage.

Soft-label multiplier **0.5** (2.5× the old 0.2); gold multiplier 1.0. Uncertain and unmentioned masks remain 0.2 and 0.0. Every weak study is used once across five epochs per fold (~350 per epoch), while gold training studies appear every epoch. Only gold validation controls checkpoint selection.

**Before running:** enable GPU; attach RSNA data, your updated 1,750-row rsna_medgemma_modal27b_soft1000.csv, and a complete local Hugging Face DINOv3 model folder. Set LOCAL_DINO_DIR to the folder containing config.json, model.safetensors and preprocessor_config.json. Model loading is local-only and needs no HF token on Kaggle. If your label export is named soft1750.csv instead, set WEAK_LABEL_CSV to its exact path; the row requirement remains 1,750.

Download on your own computer after accepting access on https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m:
```shell
python -m pip install -U huggingface_hub
hf auth login
hf download facebook/dinov3-vits16-pretrain-lvd1689m --local-dir dinov3-vits16 --include "*.json" "*.safetensors" "README.md" "LICENSE*"
```
Upload the downloaded folder as a private Kaggle dataset and attach it to this notebook.

Run the dependency cell first, then configuration and all following cells in order. Restart the kernel after installation if existing imports conflict. Offline runs need dependencies already installed or attached wheels. For competition inference, attach all five saved fold checkpoints and the original attached DINOv3 model folder, configure paths, and use the model/prediction definitions without rerunning training. The final cell writes submission.csv.

Normalization is corrected and shares the versioned image cache with the CoAtNet notebook. Outputs have unique dinov3_medgemma1750_w05 names. Training does not resume automatically. Full RSNA/GPU training has not been run during notebook creation.

Implementation reference: https://huggingface.co/docs/transformers/v4.56.2/en/model_doc/dinov3


In [1]:
%pip install -q 'transformers==4.56.2' safetensors pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg opencv-python-headless scikit-learn


ERROR: Could not find a version that satisfies the requirement transformers==4.56.2 (from versions: none)
ERROR: No matching distribution found for transformers==4.56.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor
import time, random, gc
import numpy as np
import pandas as pd
import torch, cv2, pydicom
from transformers import AutoModel, AutoConfig, AutoImageProcessor
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark=True
LABEL_COLS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
IMAGE_SIZE=224; CENTERS_PER_PLANE=8; IMAGE_MICROBATCH=2
NUM_DICOM_THREADS=8; NUM_WORKERS=2; N_FOLDS=5; EPOCHS=5
PSEUDO_LOSS_WEIGHT=.5; EXPECTED_PSEUDO_COUNT=1750
WEAK_LABEL_CSV= Path("/kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1750.csv")
LR_BACKBONE=1e-5; LR_HEAD=2e-4
DATA_CANDIDATES=[Path('/kaggle/input/rsna-knee-abnormality-detection'),Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
DATA_ROOT=next((p for p in DATA_CANDIDATES if p.exists()),None)
if DATA_ROOT is None: raise FileNotFoundError('Attach RSNA Knee Abnormality Detection competition data.')
TRAIN_SERIES_ROOT=DATA_ROOT/'train_series'; TEST_SERIES_ROOT=DATA_ROOT/'test_series'
OUT=Path('/kaggle/working'); CACHE=OUT/'rsna_2p5d_medgemma1750_norm_v2_cache'; CACHE.mkdir(exist_ok=True)
DINO_MODEL_ID='facebook/dinov3-vits16-pretrain-lvd1689m'
LOCAL_DINO_DIR=Path('/kaggle/input/models/amritgopinath/dino-v3/transformers/default/2')  # EDIT to the folder containing config.json
print('device',DEVICE)


device cuda


In [3]:
# Strictly local model files: no model download or authentication on Kaggle.
DINO_SOURCE=Path(LOCAL_DINO_DIR)
for name in ('config.json','preprocessor_config.json'):
    if not (DINO_SOURCE/name).is_file():
        raise FileNotFoundError(f'Missing {name} in {DINO_SOURCE}. Set LOCAL_DINO_DIR to the actual model folder.')
if not ((DINO_SOURCE/'model.safetensors').is_file() or (DINO_SOURCE/'model.safetensors.index.json').is_file()):
    raise FileNotFoundError(f'Missing model weights in {DINO_SOURCE}')
DINO_CONFIG=AutoConfig.from_pretrained(DINO_SOURCE,local_files_only=True)
assert DINO_CONFIG.model_type=='dinov3_vit', 'Expected DINOv3 ViT weights'
processor=AutoImageProcessor.from_pretrained(DINO_SOURCE,local_files_only=True)
DINO_MEAN=processor.image_mean; DINO_STD=processor.image_std
print('Local DINOv3 source:',DINO_SOURCE)


Local DINOv3 source: /kaggle/input/models/amritgopinath/dino-v3/transformers/default/2


In [4]:
train=pd.read_csv(DATA_ROOT/'train.csv',dtype={'StudyInstanceUID':str})
train_series=pd.read_csv(DATA_ROOT/'train_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
test_series=pd.read_csv(DATA_ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
sample_submission=pd.read_csv(DATA_ROOT/'sample_submission.csv',dtype={'StudyInstanceUID':str})
assert list(sample_submission.columns)==['StudyInstanceUID']+LABEL_COLS
gold=train[train[LABEL_COLS].notna().all(axis=1)].copy().reset_index(drop=True)
gold[LABEL_COLS]=gold[LABEL_COLS].apply(pd.to_numeric)
assert len(gold)==58, f'Expected 58 gold studies; got {len(gold)}'
assert gold.StudyInstanceUID.is_unique
print('gold studies',len(gold),'test studies',len(sample_submission))

# Load only the exact export name, never recursively scan attached inputs.
import hashlib
CSV_NAME='rsna_medgemma_modal27b_soft1000.csv'
if WEAK_LABEL_CSV is not None:
    csv_paths=[Path(WEAK_LABEL_CSV)]
else:
    root=Path('/kaggle/input')
    csv_paths=sorted(set([p for pattern in (f'*/{CSV_NAME}',f'datasets/*/*/{CSV_NAME}')
                         for p in root.glob(pattern)] + ([OUT/CSV_NAME] if (OUT/CSV_NAME).is_file() else [])))
if len(csv_paths)!=1 or not csv_paths[0].is_file():
    raise FileNotFoundError(f'Set WEAK_LABEL_CSV to the 1750-row {CSV_NAME}; found {csv_paths}')
print('Reading weak labels from',csv_paths[0])
pseudo=pd.read_csv(csv_paths[0],dtype={'StudyInstanceUID':str,'report_sha256':str})
assert len(pseudo)==EXPECTED_PSEUDO_COUNT and pseudo.StudyInstanceUID.is_unique, f'Expected {EXPECTED_PSEUDO_COUNT} unique rows; got {len(pseudo)}'
required={'StudyInstanceUID','report_sha256','model','prompt_version',*LABEL_COLS,
          *('weight__'+x for x in LABEL_COLS),*('status__'+x for x in LABEL_COLS)}
assert required.issubset(pseudo.columns), f'Missing CSV columns: {required-set(pseudo.columns)}'
assert 'Report' in train.columns and train.StudyInstanceUID.is_unique
assert set(pseudo.StudyInstanceUID).isdisjoint(set(gold.StudyInstanceUID)), 'Gold study in weak labels'
assert set(pseudo.StudyInstanceUID).issubset(set(train.StudyInstanceUID)), 'Unexpected study ID'
assert pseudo.model.eq('google/medgemma-27b-text-it').all()
assert pseudo.prompt_version.eq('medgemma-knee-v1-modal-text-v1').all()
report_lookup=train.set_index('StudyInstanceUID')['Report']
for row in pseudo[['StudyInstanceUID','report_sha256']].itertuples(index=False):
    digest=hashlib.sha256(str(report_lookup.loc[row.StudyInstanceUID]).encode('utf-8')).hexdigest()
    if row.report_sha256!=digest:
        raise ValueError(f'Report hash mismatch for study {row.StudyInstanceUID}')
soft_map={'positive':(.95,1.),'negative':(.05,1.),'uncertain':(.5,.2),'not_mentioned':(.5,0.)}
for col in LABEL_COLS:
    values=pd.to_numeric(pseudo[col],errors='raise')
    weights=pd.to_numeric(pseudo['weight__'+col],errors='raise')
    expected=pseudo['status__'+col].map(soft_map)
    if expected.isna().any(): raise ValueError(f'Invalid status for {col}')
    assert np.allclose(values.to_numpy(),expected.map(lambda v:v[0]).to_numpy())
    assert np.allclose(weights.to_numpy(),expected.map(lambda v:v[1]).to_numpy())
    pseudo[col]=values.astype(float)
    pseudo['weight__'+col]=weights.astype(float)
pseudo=pseudo.sample(frac=1,random_state=SEED).reset_index(drop=True)
print('report-only pseudo rows',len(pseudo),'usable targets',int(pseudo[['weight__'+x for x in LABEL_COLS]].gt(0).sum().sum()))


gold studies 58 test studies 3
Reading weak labels from /kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1750.csv
report-only pseudo rows 1750 usable targets 14617


In [5]:
def make_backbone(load_pretrained=True):
    net=(AutoModel.from_pretrained(DINO_SOURCE,local_files_only=True) if load_pretrained
         else AutoModel.from_config(DINO_CONFIG))
    if not hasattr(net,'layer') or not hasattr(net,'norm'):
        raise RuntimeError('Unexpected DINOv3 layout: expected layer and norm.')
    for p in net.parameters(): p.requires_grad_(False)
    for module in (net.layer[-1],net.norm):
        for p in module.parameters(): p.requires_grad_(True)
    return net

class StudyDINOv3(nn.Module):
    def __init__(self,load_pretrained=True):
        super().__init__()
        self.backbone=make_backbone(load_pretrained)
        dim=self.backbone.config.hidden_size
        self.head=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(.15),nn.Linear(dim,len(LABEL_COLS)))
        self.plane_weight=nn.Parameter(torch.zeros(4,len(LABEL_COLS)))
        self.register_buffer('mean',torch.tensor(DINO_MEAN).view(1,3,1,1))
        self.register_buffer('std',torch.tensor(DINO_STD).view(1,3,1,1))

    def train_backbone_tail(self):
        self.backbone.eval()
        self.backbone.layer[-1].train()
        self.backbone.norm.train()

    def forward(self,images,planes):
        slice_logits=[]
        for chunk in images.split(IMAGE_MICROBATCH):
            x=(chunk.to(dtype=self.mean.dtype)/255-self.mean)/self.std
            ctx=torch.autocast('cuda',dtype=torch.float16) if self.training and images.is_cuda else nullcontext()
            with ctx: slice_logits.append(self.head(self.backbone(pixel_values=x).last_hidden_state[:,0]))
        scores=torch.cat(slice_logits,0)
        pooled=[]; available=[]
        for plane in range(4):
            part=scores[planes==plane]
            available.append(len(part)>0)
            pooled.append(part.topk(min(2,len(part)),dim=0).values.mean(0) if len(part) else torch.zeros_like(scores[0]))
        mask=torch.tensor(available,device=images.device)
        weights=self.plane_weight.masked_fill(~mask[:,None],-1e4).softmax(0)
        return (weights*torch.stack(pooled)).sum(0)

# Check checkpoint compatibility before the long DICOM caching stage.
_probe_backbone=make_backbone(load_pretrained=True)
del _probe_backbone
print("DINOv3 checkpoint loaded; last block and final LayerNorm will fine-tune")


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

DINOv3 checkpoint loaded; last block and final LayerNorm will fine-tune


In [6]:
def plane_of(row):
    p=str(row.get('Anatomical_Plane','')).strip().upper()
    if p.startswith('SAG'): return 'SAGITTAL'
    if p.startswith('COR'): return 'CORONAL'
    if p.startswith('AX'): return 'AXIAL'
    return 'UNKNOWN'

def select_series(df, ids):
    df=df.copy()
    df['StudyInstanceUID']=df.StudyInstanceUID.astype(str)
    df['SeriesInstanceUID']=df.SeriesInstanceUID.astype(str)
    df=df[df.StudyInstanceUID.isin(set(map(str,ids)))].drop_duplicates(['StudyInstanceUID','SeriesInstanceUID'])
    df['plane_norm']=df.apply(plane_of,axis=1)
    for col in ('Fluid_Sensitive','Fat_Suppression'):
        if col not in df: df[col]='UNKNOWN'
        df[col]=df[col].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    rows=[]
    for sid,group in df.groupby('StudyInstanceUID',sort=True):
        group=group.sort_values('SeriesInstanceUID')
        picks=[]
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            if len(candidates): picks.append(candidates.iloc[0])
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            used=[p for p in picks if p.plane_norm==plane]
            if len(candidates)<2 or not used: continue
            first=used[0]
            rest=candidates[candidates.SeriesInstanceUID.ne(first.SeriesInstanceUID)]
            diverse=rest[(rest.Fluid_Sensitive.ne(first.Fluid_Sensitive)) | (rest.Fat_Suppression.ne(first.Fat_Suppression))]
            if len(diverse): picks.append(diverse.iloc[0])
            elif len(rest): picks.append(rest.iloc[0])
        if not picks and len(group): picks.append(group.iloc[0])
        rows.extend([{'StudyInstanceUID':sid,'SeriesInstanceUID':str(r.SeriesInstanceUID),'plane_norm':r.plane_norm} for r in picks])
    return pd.DataFrame(rows,columns=['StudyInstanceUID','SeriesInstanceUID','plane_norm'])

train_picks=select_series(train_series,pd.concat([gold.StudyInstanceUID,pseudo.StudyInstanceUID],ignore_index=True))
test_picks=select_series(test_series,sample_submission.StudyInstanceUID)
train_picks.to_csv(OUT/'coatnet_morecoverage_gold_series_selection.csv',index=False)
assert set(gold.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Gold study has no metadata series'
assert set(pseudo.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Pseudo study has no metadata series'
print('gold series',len(train_picks),'test series',len(test_picks))


gold series 8968 test series 14


In [7]:
def _read_one(path):
    try:
        d=pydicom.dcmread(str(path),force=True)
        x=np.squeeze(d.pixel_array)
        if x.ndim!=2: return None
        x=x.astype(np.float32)*float(getattr(d,'RescaleSlope',1))+float(getattr(d,'RescaleIntercept',0))
        if str(getattr(d,'PhotometricInterpretation',''))=='MONOCHROME1': x=x.max()-x
        o=np.asarray(getattr(d,'ImageOrientationPatient',[]),float)
        pos=np.asarray(getattr(d,'ImagePositionPatient',[]),float)
        g=np.nan
        if len(o)==6 and len(pos)==3:
            n=np.cross(o[:3],o[3:]); norm=np.linalg.norm(n)
            if norm>0: g=float(pos @ (n/norm))
        try: inst=float(d.InstanceNumber)
        except Exception: inst=np.nan
        return (path.name,g,inst,x)
    except Exception: return None

def resize_pad(x):
    h,w=x.shape; scale=IMAGE_SIZE/max(h,w)
    nh,nw=max(1,round(h*scale)),max(1,round(w*scale))
    z=cv2.resize(x,(nw,nh),interpolation=cv2.INTER_AREA if scale<1 else cv2.INTER_LINEAR)
    out=np.zeros((IMAGE_SIZE,IMAGE_SIZE),np.uint8)
    top=(IMAGE_SIZE-nh)//2; left=(IMAGE_SIZE-nw)//2
    out[top:top+nh,left:left+nw]=np.rint(z*255).astype(np.uint8)
    return out

def load_plane(study,series,root):
    directory=root/str(study)/str(series)
    paths=sorted(p for p in directory.iterdir() if p.is_file() and not p.name.startswith('.')) if directory.is_dir() else []
    if not paths: return None
    with ThreadPoolExecutor(max_workers=NUM_DICOM_THREADS) as pool: rows=[r for r in pool.map(_read_one,paths) if r is not None]
    if not rows: return None
    key=1 if all(np.isfinite(r[1]) for r in rows) else 2 if all(np.isfinite(r[2]) for r in rows) else 0
    rows.sort(key=lambda r:(r[key],r[0]) if key else (r[0],))
    samples=np.concatenate([r[3][::8,::8].ravel() for r in rows])
    samples=samples[np.isfinite(samples)]
    if not len(samples): return None
    lo,hi=np.percentile(samples,[1,99])
    n=len(rows); centers=np.rint(np.linspace(.05*(n-1),.95*(n-1),CENTERS_PER_PLANE)).astype(int)
    needed=sorted(set(np.clip(np.r_[centers-1,centers,centers+1],0,n-1)))
    maps={}
    for i in needed:
        x=np.nan_to_num(rows[i][3],nan=lo,posinf=hi,neginf=lo)
        x=(np.clip(x,lo,hi)-lo)/(hi-lo) if hi>lo else np.zeros_like(x)
        maps[i]=resize_pad(x.astype(np.float32))
    return np.stack([np.stack([maps[max(0,i-1)],maps[i],maps[min(n-1,i+1)]],0) for i in centers],0)

def cache_path(study,root):
    return CACHE/('train_' if root==TRAIN_SERIES_ROOT else 'test_')/f'{study}.npz'

def cache_study(study,picks,root):
    path=cache_path(study,root)
    if path.exists(): return path
    blocks=[]; planes=[]
    group=picks[picks.StudyInstanceUID.eq(str(study))]
    for row in group.itertuples(index=False):
        try: block=load_plane(study,row.SeriesInstanceUID,root)
        except Exception: block=None
        if block is None: continue
        blocks.append(block); planes.extend([('SAGITTAL','CORONAL','AXIAL').index(row.plane_norm) if row.plane_norm in ('SAGITTAL','CORONAL','AXIAL') else 3]*len(block))
    if not blocks: return None
    path.parent.mkdir(parents=True,exist_ok=True)
    np.savez(path,images=np.concatenate(blocks,0),planes=np.asarray(planes,np.int8))
    return path

def warm_cache(ids,picks,root):
    t=time.time(); failures=[]
    for i,sid in enumerate(map(str,ids),1):
        if cache_study(sid,picks,root) is None: failures.append(sid)
        if i%100==0 or i==len(ids): print(f'cached {i}/{len(ids)} in {(time.time()-t)/60:.1f} min; failed={len(failures)}')
    return failures


gold_failures=warm_cache(gold.StudyInstanceUID.tolist(),train_picks,TRAIN_SERIES_ROOT)
assert not gold_failures, f'Gold studies failed DICOM decode: {gold_failures}'
pseudo_failures=warm_cache(pseudo.StudyInstanceUID.tolist(),train_picks,TRAIN_SERIES_ROOT)
if pseudo_failures: print('Dropped undecodable pseudo studies:',len(pseudo_failures))
pseudo=pseudo.loc[~pseudo.StudyInstanceUID.isin(pseudo_failures)].reset_index(drop=True)
assert len(pseudo)>0, 'No pseudo studies decoded.'


cached 58/58 in 1.1 min; failed=0
cached 100/1750 in 2.2 min; failed=0
cached 200/1750 in 4.5 min; failed=0
cached 300/1750 in 6.7 min; failed=0
cached 400/1750 in 9.0 min; failed=0
cached 500/1750 in 11.2 min; failed=0
cached 600/1750 in 13.6 min; failed=0
cached 700/1750 in 15.9 min; failed=0
cached 800/1750 in 18.1 min; failed=0
cached 900/1750 in 20.1 min; failed=0
cached 1000/1750 in 22.5 min; failed=0
cached 1100/1750 in 24.7 min; failed=0
cached 1200/1750 in 26.8 min; failed=0
cached 1300/1750 in 28.7 min; failed=0
cached 1400/1750 in 30.6 min; failed=0
cached 1500/1750 in 32.7 min; failed=0
cached 1600/1750 in 34.8 min; failed=0
cached 1700/1750 in 36.9 min; failed=0
cached 1750/1750 in 37.9 min; failed=0


In [8]:
class BagDS(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with np.load(cache_path(str(row.StudyInstanceUID),TRAIN_SERIES_ROOT)) as d:
            x=d['images'].copy(); plane=d['planes'].astype(np.int64)
        return torch.from_numpy(x),torch.from_numpy(plane),torch.tensor(row[LABEL_COLS].to_numpy(np.float32))

class PseudoBagDS(BagDS):
    def __getitem__(self,i):
        x,plane,y=super().__getitem__(i)
        row=self.frame.iloc[i]
        weight=torch.tensor(row[['weight__'+c for c in LABEL_COLS]].to_numpy(np.float32))
        return x,plane,y,weight

def train_fold(model,frame,validation,fold,checkpoint):
    gold_loader=DataLoader(BagDS(frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                      collate_fn=lambda b:b[0])
    print("trainable backbone parameters",sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
    optimizer=torch.optim.AdamW([{'params':[p for p in model.backbone.parameters() if p.requires_grad],'lr':LR_BACKBONE},
                                 {'params':list(model.head.parameters())+[model.plane_weight],'lr':LR_HEAD}],weight_decay=1e-4)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
    best_bce=np.inf; best_epoch=None; history=[]
    # The same pseudo order is shared by folds, but the model and gold split differ.
    order=np.random.default_rng(SEED).permutation(len(pseudo))
    weak_chunks=np.array_split(order,EPOCHS)
    for epoch in range(1,EPOCHS+1):
        model.train(); model.train_backbone_tail()
        losses=[]; start=time.time()
        weak_frame=pseudo.iloc[weak_chunks[epoch-1]]
        assert len(weak_frame)>0
        weak_loader=DataLoader(PseudoBagDS(weak_frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                          collate_fn=lambda b:b[0])
        gold_iter=iter(gold_loader); weak_iter=iter(weak_loader)
        schedule=np.array([0]*len(frame)+[1]*len(weak_frame))
        np.random.default_rng(SEED+fold*100+epoch).shuffle(schedule)
        for mode in schedule:
            if mode==0:
                x,plane,y=next(gold_iter); weights=None
            else:
                x,plane,y,weights=next(weak_iter)
            x=x.to(DEVICE,non_blocking=True); plane=plane.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,plane).float()
            if mode==0:
                loss=nn.functional.binary_cross_entropy_with_logits(logits,y)
            else:
                weights=weights.to(DEVICE)
                raw=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
                # Preserve uncertain/not-mentioned masking and don't inflate a study with few asserted findings.
                loss=PSEUDO_LOSS_WEIGHT*(raw*weights).sum()/weights.numel()
            if not torch.isfinite(loss): raise RuntimeError(f'Nonfinite loss in fold {fold}, epoch {epoch}')
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            losses.append(loss.item())
        pred,failed=predict_ids(model,validation.StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
        if failed: raise RuntimeError(f'Fold {fold} validation decode failures: {failed}')
        aligned=validation[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
        y=validation[LABEL_COLS].to_numpy(np.float32)
        probs=aligned[LABEL_COLS].to_numpy(np.float32).clip(1e-6,1-1e-6)
        val_bce=nn.functional.binary_cross_entropy(torch.from_numpy(probs),torch.from_numpy(y)).item()
        aucs=[roc_auc_score(y[:,j],probs[:,j]) for j in range(len(LABEL_COLS)) if np.unique(y[:,j]).size==2]
        val_auc=float(np.mean(aucs)) if aucs else float('nan')
        if val_bce<best_bce-1e-6:
            best_bce=val_bce; best_epoch=epoch
            torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},checkpoint)
        history.append({'fold':fold,'epoch':epoch,'train_weighted_loss':float(np.mean(losses)),
                        'val_bce':val_bce,'val_macro_auc':val_auc,'n_auc_labels':len(aucs),
                        'best_so_far':best_epoch,'minutes':(time.time()-start)/60})
        print(f'fold {fold} epoch {epoch}/{EPOCHS}: weighted loss {np.mean(losses):.4f}, '
              f'val BCE {val_bce:.4f}, val AUC {val_auc:.4f} ({len(aucs)} labels), '
              f'best epoch {best_epoch}, time {(time.time()-start)/60:.1f}m')
    model.load_state_dict(torch.load(checkpoint,map_location='cpu',weights_only=True),strict=True)
    return best_epoch,best_bce,history

@torch.inference_mode()
def predict_ids(model,ids,picks,root):
    model.eval(); rows=[]; failed=[]
    for sid in map(str,ids):
        path=cache_path(sid,root)
        if not path.exists(): cache_study(sid,picks,root)
        if not path.exists(): failed.append(sid); continue
        with np.load(path) as d:
            x=torch.from_numpy(d['images'].copy()).to(DEVICE); plane=torch.from_numpy(d['planes'].astype(np.int64)).to(DEVICE)
        score=torch.sigmoid(model(x,plane).float()).cpu().numpy()
        rows.append([sid,*score.tolist()])
    return pd.DataFrame(rows,columns=['StudyInstanceUID']+LABEL_COLS),failed

folds=KFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
oof=np.full((len(gold),len(LABEL_COLS)),np.nan,np.float32)
fold_scores=[]; epoch_history=[]
for fold,(tr,va) in enumerate(folds.split(gold),1):
    print('\nFOLD',fold,'train',len(tr),'validation',len(va))
    model=StudyDINOv3(load_pretrained=True).to(DEVICE)
    checkpoint=OUT/f'dinov3_medgemma1750_w05_fold{fold}.pt'
    best_epoch,best_bce,history=train_fold(model,gold.iloc[tr],gold.iloc[va],fold,checkpoint)
    epoch_history.extend(history)
    pred,failed=predict_ids(model,gold.iloc[va].StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
    assert not failed
    aligned=gold.iloc[va][['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
    oof[va]=aligned[LABEL_COLS].to_numpy(np.float32)
    fold_scores.append({'fold':fold,'n_train':len(tr),'n_validation':len(va),'checkpoint':str(checkpoint),'best_epoch':best_epoch,'best_val_bce':best_bce})
    print('saved',checkpoint)
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
assert np.isfinite(oof).all()
oof_df=gold[['StudyInstanceUID']+LABEL_COLS].copy()
for j,label in enumerate(LABEL_COLS): oof_df['pred__'+label]=oof[:,j]
oof_df.to_csv(OUT/'dinov3_medgemma1750_w05_oof_58.csv',index=False)
pd.DataFrame(fold_scores).to_csv(OUT/'dinov3_medgemma1750_w05_fold_manifest.csv',index=False)
pd.DataFrame(epoch_history).to_csv(OUT/'dinov3_medgemma1750_w05_epoch_history.csv',index=False)
metrics=[]
for j,label in enumerate(LABEL_COLS):
    y=gold[label].to_numpy(float)
    metrics.append({'label':label,'positives':int(y.sum()),'auc':roc_auc_score(y,oof[:,j]) if np.unique(y).size==2 else np.nan})
metrics=pd.DataFrame(metrics); metrics.to_csv(OUT/'dinov3_medgemma1750_w05_oof_auc.csv',index=False)
display(metrics); print('5-fold OOF macro AUC:',metrics.auc.mean())



FOLD 1 train 46 validation 12


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

trainable backbone parameters 1775616


/tmp/ipykernel_22/1772835188.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 1 epoch 1/5: weighted loss 0.2644, val BCE 0.5876, val AUC 0.5906 (10 labels), best epoch 1, time 2.8m
fold 1 epoch 2/5: weighted loss 0.2543, val BCE 0.5808, val AUC 0.5696 (10 labels), best epoch 2, time 2.7m
fold 1 epoch 3/5: weighted loss 0.2561, val BCE 0.6311, val AUC 0.5762 (10 labels), best epoch 2, time 2.7m
fold 1 epoch 4/5: weighted loss 0.2485, val BCE 0.6399, val AUC 0.5719 (10 labels), best epoch 2, time 2.7m
fold 1 epoch 5/5: weighted loss 0.2460, val BCE 0.5974, val AUC 0.5437 (10 labels), best epoch 2, time 2.7m
saved /kaggle/working/dinov3_medgemma1750_w05_fold1.pt

FOLD 2 train 46 validation 12


...

trainable backbone parameters 1775616


/tmp/ipykernel_22/1772835188.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 5 epoch 1/5: weighted loss 0.2657, val BCE 0.6936, val AUC 0.5165 (12 labels), best epoch 1, time 2.6m
fold 5 epoch 2/5: weighted loss 0.2546, val BCE 0.6921, val AUC 0.5858 (12 labels), best epoch 2, time 2.5m
fold 5 epoch 3/5: weighted loss 0.2565, val BCE 0.6675, val AUC 0.6137 (12 labels), best epoch 3, time 2.5m
fold 5 epoch 4/5: weighted loss 0.2494, val BCE 0.6673, val AUC 0.6192 (12 labels), best epoch 4, time 2.6m
fold 5 epoch 5/5: weighted loss 0.2449, val BCE 0.6624, val AUC 0.6480 (12 labels), best epoch 5, time 2.5m
saved /kaggle/working/dinov3_medgemma1750_w05_fold5.pt


,label,positives,auc
0,ACL,24,0.572304
1,MCL,9,0.718821
2,Medial Meniscus,26,0.652644
3,Lateral Meniscus,23,0.571429
4,Medial OA,15,0.638760
5,Lateral OA,11,0.603482
6,PF OA,21,0.662162
7,Effusion,35,0.734161
8,Synovitis,27,0.457587
9,Baker's,12,0.545290


5-fold OOF macro AUC: 0.601563523648336


In [9]:
# The test DICOMs are decoded and cached once; the five fold checkpoints are used sequentially.
test_ids=sample_submission.StudyInstanceUID.astype(str).tolist()
test_failures=warm_cache(test_ids,test_picks,TEST_SERIES_ROOT)
print('test decode failures',len(test_failures))
ensemble=np.zeros((len(test_ids),len(LABEL_COLS)),np.float64)
for fold in range(1,N_FOLDS+1):
    model=StudyDINOv3(load_pretrained=False)
    state=torch.load(OUT/f'dinov3_medgemma1750_w05_fold{fold}.pt',map_location='cpu',weights_only=True)
    model.load_state_dict(state,strict=True)
    model=model.to(DEVICE)
    if DEVICE.type=='cuda': model=model.half()  # half weights and half inputs, no autocast at inference
    pred,failed=predict_ids(model,test_ids,test_picks,TEST_SERIES_ROOT)
    aligned=sample_submission[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',how='left',validate='one_to_one')
    fallback=gold[LABEL_COLS].mean().to_numpy(float)
    for j,label in enumerate(LABEL_COLS): aligned[label]=aligned[label].fillna(float(fallback[j]))
    ensemble+=aligned[LABEL_COLS].to_numpy(float)/N_FOLDS
    print('fold',fold,'predicted',len(pred),'missing',len(failed))
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
submission=sample_submission[['StudyInstanceUID']].copy()
submission[LABEL_COLS]=ensemble.astype(np.float32)
assert list(submission.columns)==list(sample_submission.columns)
assert submission.StudyInstanceUID.is_unique and len(submission)==len(sample_submission)
assert np.isfinite(ensemble).all() and ensemble.min()>=0 and ensemble.max()<=1
submission.to_csv(OUT/'submission.csv',index=False)
print('wrote',OUT/'submission.csv','shape',submission.shape)
display(submission.head())


cached 3/3 in 0.1 min; failed=0
test decode failures 0
fold 1 predicted 3 missing 0
fold 2 predicted 3 missing 0
fold 3 predicted 3 missing 0
fold 4 predicted 3 missing 0
fold 5 predicted 3 missing 0
wrote /kaggle/working/submission.csv shape (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.251373,0.207614,0.511168,0.323125,0.549211,0.418851,0.616431,0.542997,0.562481,0.445249,0.461149,0.212734
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.246680,0.177511,0.456812,0.297820,0.512674,0.327276,0.547409,0.538707,0.532348,0.374626,0.449869,0.213210
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.247732,0.141674,0.431827,0.296643,0.423738,0.318845,0.502658,0.481602,0.457554,0.316408,0.434650,0.205365
